In [1]:
import numpy as np
import pandas as pd
import joblib
import os
from sklearn.metrics import r2_score, mean_squared_error

# -----------------------------
# Configuración
# -----------------------------
models = ["ridge", "rf", "mlp", "xgb"]  # Incluye todos los modelos disponibles

# -----------------------------
# Cargar datos test
# -----------------------------
X_test = np.load('../outputs/X_test.npy')
y_test = np.load('../outputs/y_test.npy')
df_real = pd.read_csv("../data/processed_dataset.csv")  # Mantener 'seq' original

# -----------------------------
# Evaluar modelos
# -----------------------------
results = []
df_preds = pd.DataFrame()
df_preds["seq"] = df_real["seq"][:len(X_test)]  # asegurar match de tamaño

for name in models:
    print(f"Evaluando {name}...")
    
    model_path = f'../outputs/model_{name}.pkl'
    if not os.path.exists(model_path):
        print(f"⚠️ Modelo {name} no encontrado en {model_path}, se salta.")
        continue
    
    
    model = joblib.load(model_path)
    preds = model.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    results.append({
        "model": name,
        "R2": r2,
        "RMSE": rmse
    })
    
    df_preds[f"{name}_PTM"] = preds[:,0]
    df_preds[f"{name}_IPTM"] = preds[:,1]

# -----------------------------
# Predicción promedio de todos los modelos
# -----------------------------
ptm_cols = [col for col in df_preds.columns if "_PTM" in col]
iptm_cols = [col for col in df_preds.columns if "_IPTM" in col]

df_preds["PTM_pred"] = df_preds[ptm_cols].mean(axis=1)
df_preds["IPTM_pred"] = df_preds[iptm_cols].mean(axis=1)

# -----------------------------
# Guardar predicciones y métricas
# -----------------------------
os.makedirs("../outputs", exist_ok=True)
df_preds.to_csv('../outputs/predictions_all_models.csv', index=False)
pd.DataFrame(results).to_csv('../outputs/model_metrics.csv', index=False)

print("\nEvaluación completada. Predicciones guardadas en '../outputs/predictions_all_models.csv'")
print(df_preds.head())

Evaluando ridge...
Evaluando rf...


[Parallel(n_jobs=30)]: Using backend ThreadingBackend with 30 concurrent workers.
[Parallel(n_jobs=30)]: Done   2 out of  30 | elapsed:    0.1s remaining:    2.1s
[Parallel(n_jobs=30)]: Done  30 out of  30 | elapsed:    0.2s finished
[Parallel(n_jobs=30)]: Using backend ThreadingBackend with 30 concurrent workers.
[Parallel(n_jobs=30)]: Done   2 out of  30 | elapsed:    0.3s remaining:    4.5s
[Parallel(n_jobs=30)]: Done  30 out of  30 | elapsed:    0.3s finished


Evaluando mlp...
Evaluando xgb...

Evaluación completada. Predicciones guardadas en '../outputs/predictions_all_models.csv'
                                                 seq  ridge_PTM  ridge_IPTM  \
0  GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...   0.728508    0.216338   
1  GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...   0.725126    0.201112   
2  GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...   0.738307    0.237314   
3  GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...   0.726664    0.213103   
4  GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...   0.666207    0.167557   

     rf_PTM   rf_IPTM   mlp_PTM  mlp_IPTM   xgb_PTM  xgb_IPTM  PTM_pred  \
0  0.724357  0.194741  0.722001  0.206778  0.722710  0.211369  0.724394   
1  0.726381  0.199345  0.715453  0.165389  0.718732  0.183301  0.721423   
2  0.725516  0.187702  0.728508  0.199154  0.726913  0.187870  0.729811   
3  0.720437  0.182318  0.724970  0.262951  0.724641  0.213451  0.724178   
4  0.685174  0.229969  0.6